# Búsqueda de tamaño y mezcla de DART + CRNN + HMM

**Qué significa una ronda.** Dividimos las 34 personas en cinco grupos, conservando juntas todas las noches de cada persona. Cada grupo ocupa una vez el lugar de prueba. Al unir esas cinco pruebas se obtiene una predicción fuera de entrenamiento para cada uno de los **75.612 segmentos de 30 s**. El F1 macro global se calcula sobre esa unión, dando el mismo peso a W, N1, N2, N3 y REM.

**Qué se elige.** En cada ronda, un segundo grupo sirve para elegir el tamaño de la CRNN y la mezcla mediante F1 macro. DART, CRNN y HMM internos se entrenan únicamente en los otros tres grupos. Después se usan los modelos ajustados sobre los cuatro grupos disponibles para evaluar el quinto. El DART externo ya estaba entrenado para esa misma partición. Los empates internos favorecen la primera red (de menor a mayor tamaño), y dentro de una red el porcentaje de DART más alto.

La comparación usa un solo LightGBM (DART). La CRNN clasifica los segmentos, el HMM procesa sus probabilidades a lo largo de cada noche y por último se combina con DART. Las cuatro redes comparten datos, orden de minibatches, 12 épocas, Adam 0,001, lote 64, dropout 0,15/0,20, contexto de 15 segmentos y preprocesamiento espectral de seis bandas por canal. Un generador NumPy con semilla 42 fija el orden de entrenamiento por época para todas las candidatas. TensorFlow también se inicializa con esa semilla y se solicitan operaciones deterministas.

Estas cohortes se han usado en experimentos anteriores. El resultado describe validación por persona sobre este dataset de desarrollo. Una cohorte independiente permitiría evaluar su generalización.

**Resultado sobre todos los datos disponibles.** La selección interna por persona obtuvo **81,01 % de F1 macro, 86,81 % de exactitud y kappa 0,8177**. DART solo obtuvo 80,10 % de F1. La selección eligió la red amplia en tres rondas y la mediana en dos, con pesos DART entre 65 % y 75 %. En Telemetry el F1 pasó de 78,81 % a 80,17 %. En Cassette pasó de 77,68 % a 77,12 %.

**Para usar una sola configuración**, el mayor F1 observado entre las mezclas fijas fue **80,91 %** con **75 % DART + 25 % CRNN amplia con HMM**. Con esa misma red, el 70/30 alcanzó 80,87 %. La diferencia es de 0,04 puntos de F1. La CRNN amplia tiene 62.629 parámetros y tardó entre 25 y 38 segundos por ajuste externo. Cuando se comunique el 81,01 %, identificarlo como rendimiento de la regla de selección por ronda.

In [1]:
import os
import sys
from pathlib import Path
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'data.dvc').is_file())
sys.path.insert(0,str(root))
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
from scripts.buscar_fusion_crnn import (
    CLASES, TAMANOS, VALIDACION, preparar, entrenar_redes, comparar, metricas
)
import pandas as pd
from sklearn.metrics import confusion_matrix
meta, dart, noches, s_dart, s_senal = preparar()
display(meta.groupby('cohorte').agg(personas=('subject_id','nunique'), noches=('record_id','nunique'), segmentos=('real','size')))
display(pd.DataFrame.from_dict(TAMANOS,orient='index',columns=['Filtros Conv1D','Unidades densas','GRU por dirección']))
particiones = []
for ronda, interna in VALIDACION.items():
    for uso, mascara in [('Ajuste interno',~meta.ronda.isin([ronda,interna])),('Validación interna',meta.ronda.eq(interna)),
                         ('Entrenamiento final',meta.ronda.ne(ronda)),('Prueba externa',meta.ronda.eq(ronda))]:
        g = meta[mascara]
        particiones.append([ronda,uso,g.subject_id.nunique(),g.record_id.nunique(),len(g)])
display(pd.DataFrame(particiones,columns=['Ronda','Uso','Personas','Noches','Segmentos']).set_index(['Ronda','Uso']))

,personas,noches,segmentos
cohorte,,,
Cassette,12,12,32921
Telemetry,22,44,42691


,Filtros Conv1D,Unidades densas,GRU por dirección
Mini,16,16,8
Compacta,32,32,16
Mediana,48,48,32
Amplia,64,64,48


Personas  Noches  Segmentos
Ronda Uso                                             
1     Ajuste interno             20      34      43956
      Validación interna          7      11      15763
      Entrenamiento final        27      45      59719
      Prueba externa              7      11      15893
2     Ajuste interno             20      34      43956
      Validación interna          7      11      15893
      Entrenamiento final        27      45      59849
      Prueba externa              7      11      15763
3     Ajuste interno             20      32      45214
      Validación interna          7      12      14989
      Entrenamiento final        27      44      60203
      Prueba externa              7      12      15409
4     Ajuste interno             20      32      45214
      Validación interna          7      12      15409
      Entrenamiento final        27      44      60623
      Prueba externa              7      12      14989
5     Ajuste interno             21      35      46161
      Validación interna          7      11      15893
      Entrenamiento final        28      46      62054
      Prueba externa              6      10      13558

## Resultado de la selección interna

La fila «Selección interna» evalúa el procedimiento completo de elegir la configuración usando solo el entrenamiento. Las filas 75/25 y 70/30 son mezclas fijas con la red compacta. La nueva corrida fija explícitamente el orden de los minibatches y por eso sus valores pueden diferir del ensayo histórico.

In [2]:
cache = entrenar_redes(meta, noches)
oof, elegidos, combinaciones, tiempos = comparar(meta, dart, cache)
oof.to_parquet(cache/'predicciones_seleccion.parquet', index=False)
display(elegidos.round(4))
display(pd.concat({'Ambas cohortes':metricas(oof), **{k:metricas(g) for k,g in oof.groupby('cohorte')}}).round(4))
print('Resultados de prueba de cada grupo con selección interna')
display(pd.DataFrame({k:metricas(g).loc['Selección interna'] for k,g in oof.groupby('ronda')}).T.round(4))

,Ronda,Validación interna,CRNN,% DART,F1 interno
0,1,2,Amplia,70,0.7979
1,2,1,Amplia,75,0.8293
2,3,4,Mediana,70,0.8001
3,4,3,Mediana,75,0.8026
4,5,1,Amplia,65,0.8422


Segmentos  F1 macro  Exactitud   Kappa
               Modelo                                                   
Ambas cohortes DART                   75612    0.8010     0.8596  0.8074
               Compacta 75/25         75612    0.8086     0.8675  0.8171
               Compacta 70/30         75612    0.8071     0.8673  0.8165
               Selección interna      75612    0.8101     0.8681  0.8177
Cassette       DART                   32921    0.7768     0.9144  0.8233
               Compacta 75/25         32921    0.7781     0.9164  0.8254
               Compacta 70/30         32921    0.7731     0.9148  0.8216
               Selección interna      32921    0.7712     0.9117  0.8154
Telemetry      DART                   42691    0.7881     0.8174  0.7474
               Compacta 75/25         42691    0.7977     0.8298  0.7618
               Compacta 70/30         42691    0.7968     0.8306  0.7622
               Selección interna      42691    0.8017     0.8344  0.7673

Resultados de prueba de cada grupo con selección interna


,Segmentos,F1 macro,Exactitud,Kappa
1,15893.0,0.8295,0.8860,0.8414
2,15763.0,0.8099,0.8860,0.8365
3,15409.0,0.8085,0.8461,0.7903
4,14989.0,0.8113,0.8650,0.8180
5,13558.0,0.7870,0.8547,0.7950


## F1 de todas las mezclas fijas

Cada celda resume las cinco rondas completas. La columna indica el porcentaje de DART y el resto corresponde a CRNN + HMM. El 100 % reproduce el mismo DART en todas las filas. El máximo de esta tabla es exploratorio porque se compararon todas las combinaciones sobre las mismas personas. La estimación principal está en «Selección interna».

In [3]:
from IPython.display import display
display(combinaciones.pivot(index='CRNN',columns='% DART',values='F1 macro').reindex(TAMANOS).round(4))
display(combinaciones.sort_values('F1 macro',ascending=False).head(8).round(4))
print('Matriz global de la selección interna. Filas reales, columnas predichas.')
display(pd.DataFrame(confusion_matrix(oof.real,oof['Selección interna'],labels=CLASES),index=CLASES,columns=CLASES))
from sklearn.metrics import precision_recall_fscore_support
p_clase,r_clase,f_clase,soporte = precision_recall_fscore_support(
    oof.real, oof['Selección interna'], labels=CLASES, zero_division=0)
display(pd.DataFrame({'Segmentos':soporte, 'Precisión':p_clase, 'Sensibilidad':r_clase, 'F1':f_clase},index=CLASES).round(4))
assert abs(f_clase.mean()-metricas(oof).loc['Selección interna','F1 macro']) < 1e-12

% DART,0,25,40,50,60,65,70,75,80,85,90,100
CRNN,,,,,,,,,,,,
Mini,0.7293,0.7377,0.7475,0.7601,0.7943,0.8009,0.8045,0.8059,0.8066,0.8065,0.8047,0.801
Compacta,0.7469,0.7531,0.7605,0.7720,0.7984,0.8039,0.8071,0.8086,0.8081,0.8072,0.8051,0.801
Mediana,0.7477,0.7543,0.7613,0.7727,0.8002,0.8056,0.8079,0.8089,0.8090,0.8076,0.8051,0.801
Amplia,0.7473,0.7537,0.7601,0.7723,0.8018,0.8067,0.8087,0.8091,0.8084,0.8076,0.8055,0.801


,CRNN,% DART,F1 macro,Exactitud,Kappa
40,Amplia,75,0.8091,0.8675,0.8171
27,Mediana,80,0.8090,0.8669,0.8165
28,Mediana,75,0.8089,0.8673,0.8167
41,Amplia,70,0.8087,0.8676,0.8169
16,Compacta,75,0.8086,0.8675,0.8171
39,Amplia,80,0.8084,0.8666,0.8162
15,Compacta,80,0.8081,0.8666,0.8161
29,Mediana,70,0.8079,0.8670,0.8160


Matriz global de la selección interna. Filas reales, columnas predichas.


,N1,N2,N3,REM,W
N1,2728,912,17,411,791
N2,1009,21686,1920,670,743
N3,11,1051,6281,3,46
REM,292,879,14,8985,207
W,739,212,10,38,25957


,Segmentos,Precisión,Sensibilidad,F1
N1,4859,0.5708,0.5614,0.5661
N2,26028,0.8766,0.8332,0.8543
N3,7392,0.7621,0.8497,0.8035
REM,10377,0.8890,0.8659,0.8773
W,26956,0.9356,0.9629,0.9491


## Tiempo y tamaño

El ajuste mide la CRNN sobre entradas preparadas. La inferencia incluye la CRNN y el HMM sobre las noches reservadas de cada ronda. Las mediciones dependen de la CPU y de los otros ajustes concurrentes. La búsqueda prueba varios modelos y tiene un coste distinto al ajuste de una sola combinación.

In [4]:
display(tiempos.groupby('CRNN').agg(parametros=('Parámetros','first'),
    ajuste_min_s=('Ajuste CRNN (s)','min'), ajuste_max_s=('Ajuste CRNN (s)','max'),
    inferencia_media_por_ronda_s=('Inferencia CRNN + HMM (s)','mean')).reindex(TAMANOS).round(2))
import json
from scripts.evaluar_v6_compacto import JUSTA
tiempo_dart = json.loads((JUSTA/'tiempos-ronda-1/mediciones.json').read_text())['ajuste_dart_s']
nombre = elegidos.loc[elegidos.Ronda.eq(1),'CRNN'].item()
s_red = tiempos.loc[tiempos.Ronda.eq(1)&tiempos.CRNN.eq(nombre),'Ajuste CRNN (s)'].item()
print(f'Ronda 1 con entradas preparadas: DART {tiempo_dart:.1f} s (medición previa), CRNN {nombre} {s_red:.1f} s. Suma de ambos ajustes {(tiempo_dart+s_red)/60:.2f} min. El cálculo de las transiciones se mide aparte en el experimento de referencia.')

,parametros,ajuste_min_s,ajuste_max_s,inferencia_media_por_ronda_s
CRNN,,,,
Mini,4053,10.83,18.60,1.96
Compacta,13477,14.40,22.23,1.83
Mediana,33445,20.35,30.18,1.94
Amplia,62629,25.34,38.22,1.86


Ronda 1 con entradas preparadas: DART 420.7 s (medición previa), CRNN Amplia 37.0 s. Suma de ambos ajustes 7.63 min. El cálculo de las transiciones se mide aparte en el experimento de referencia.
